# 03 · GAN, part 2: generate the extra images and check them

Part 1 (`03_gan_train`) trained a class-conditional StyleGAN2-ADA on the train split. This notebook takes its **best
snapshot** (lowest FID) and:

1. generates new leaf images for every class with fewer than 2,000 training images, until the class has 2,000
   (Yellow leaf curl virus, with 3,747, gets none),
2. keeps a generated image only if the best Phase 2 model (notebook 02's ResNet50) recognises it as its class,
3. checks the kept images in four ways: a sample sheet to look at, FID and KID per class, the nearest real image
   (is the GAN copying?), and the label check's numbers,
4. saves them as `synthetic/` with the list `synthetic/synthetic.csv`, which notebook 04 reads.

| Class | Train images | To generate |
|---|---|---|
| Bacterial spot | 1,497 | 503 |
| Late blight | 1,330 | 670 |
| Septoria leaf spot | 1,235 | 765 |
| Spider mites | 1,169 | 831 |
| Healthy | 1,114 | 886 |
| Target spot | 979 | 1,021 |
| Early blight | 697 | 1,303 |
| Leaf mold | 669 | 1,331 |
| Mosaic virus | 261 | 1,739 |

The validation images are used only in check 3, as examples of new leaves; nothing is chosen with them. The test
images are not used.

**How to run it on Kaggle**

- **Add Input** → `plantvillage-dataset` (by *abdallahalidev*).
- **Add Input** → **Your Work** → **Notebooks** → notebook 01 (`splits.csv`), notebook 02 (the final run, for
  `resnet50.pt`) and **notebook 03_gan_train** (its latest version: `gan/progress.json` and `gan/snapshots/`).
- **Settings**: Accelerator **GPU T4 x2** or **P100** (one GPU is used). **Internet on**: it downloads NVIDIA's code
  (needed to read the snapshot) and the Inception network for FID and KID.
- **Save Version → Save & Run All (Commit)**. About 20–40 minutes.
- Download `results/` into the project folder, into `results-03/` with part 1's files (the names do not clash). The
  images stay on Kaggle: notebook 04 reads them from this notebook's output, and notebook 05 copies a few into the app.

In [ ]:
import os, re, sys, json, time, shutil, pickle, fnmatch, hashlib, subprocess, importlib.util
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torchvision import models

NOTEBOOK_START = time.time()

SNAPSHOT_CHOICE = 'best'   # 'best': part 1's snapshot with the lowest FID. 'latest': its last one
FILL_TARGET = 2000         # every class with fewer training images is filled up to this many
TRUNCATION_PSI = 1.0       # 1 = the full variety the GAN has learned (lower values give more typical, less varied leaves)
MAX_TRIES = 3              # generate at most 3 x the needed number per class
GEN_BATCH = 32
JPEG_QUALITY = 95          # generated images are saved as JPEG, like the PlantVillage photos
SEED = 0
RES = 256

# NVIDIA's code, the same version and changes as part 1 (needed to read its snapshots)
STYLEGAN_REPO = 'https://github.com/NVlabs/stylegan3.git'
STYLEGAN_COMMIT = 'c233a919a6faee6e36a316ddd4eddababad1adf9'
PROGRESS_KIND = 'tomato-leaf stylegan2-ada'   # marks part 1's progress.json among the inputs
SPLIT_FINGERPRINT = '7cf95c207559517a'

INPUT_ROOT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))
OUT_DIR = Path(os.environ.get('OUT_DIR', '/kaggle/working'))
WORK_DIR = Path(os.environ.get('WORK_DIR', '/tmp/gan'))   # not saved: the GAN code
RESULTS_OUT, SYNTH_OUT = OUT_DIR / 'results', OUT_DIR / 'synthetic'
for folder in (RESULTS_OUT, WORK_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    try:
        nn.functional.conv2d(torch.ones(1, 1, 4, 4, device=DEVICE), torch.ones(1, 1, 3, 3, device=DEVICE))
    except RuntimeError as e:
        raise RuntimeError(f'The GPU {torch.cuda.get_device_name()} does not work with this PyTorch ({e}). '
                           'Choose Settings → Accelerator → GPU T4 x2 and run again.') from None
    print('GPU:', torch.cuda.get_device_name(), '| PyTorch', torch.__version__)
elif Path('/kaggle').exists():
    raise RuntimeError('No GPU. Choose Settings → Accelerator → GPU T4 x2 and run again.')
else:
    DEVICE = torch.device('cpu')
    print('No GPU: running on the CPU (only for a short test run on another computer)')

# Chart style of the project (as in notebooks 01 and 02)
SURFACE, INK, INK_2, GRID, BLUE, ORANGE = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#2a78d6', '#eb6834'
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': INK_2, 'xtick.color': INK_2, 'ytick.color': INK_2,
    'axes.edgecolor': GRID, 'axes.linewidth': 1, 'grid.color': GRID, 'grid.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

## 1. The inputs

The split and the class list from notebook 01, the label-check model from notebook 02 (the final run's
`resnet50.pt`: ResNet50 was the best model on the validation split), and the GAN from part 1 (`gan/progress.json`
names its best and latest snapshots).

In [ ]:
def find_files(root, pattern):
    """Files under root whose name matches pattern. Class folders (names with ___) only hold images, so they are skipped."""
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        dirnames[:] = [d for d in dirnames if '___' not in d]
        found += [Path(dirpath) / f for f in sorted(filenames) if fnmatch.fnmatch(f, pattern)]
    return found

def find_color_dir(root):
    """PlantVillage's `color` folder (the one that holds the Tomato___ class folders) under root."""
    for dirpath, dirnames, _ in os.walk(root):
        dirnames.sort()
        if Path(dirpath).name.lower() == 'color' and any(d.startswith('Tomato') for d in dirnames):
            return Path(dirpath)
        dirnames[:] = [d for d in dirnames if '___' not in d]
    raise FileNotFoundError(f'No PlantVillage "color" folder under {root}. '
                            'Add the Kaggle dataset abdallahalidev/plantvillage-dataset as input.')

def fingerprint(splits):
    return hashlib.sha256('\n'.join(sorted(splits['path'] + ',' + splits['split'])).encode()).hexdigest()[:16]

found = {f: pd.read_csv(f) for f in find_files(INPUT_ROOT, 'splits.csv')}
final_split = [f for f, s in found.items() if fingerprint(s) == SPLIT_FINGERPRINT]
if not final_split:
    raise FileNotFoundError('The final splits.csv of notebook 01 was not found. Add Input → Your Work → Notebooks → '
                            'notebook 01, the version that made the 12,698 / 2,728 / 2,734 split.')
SPLITS_CSV = final_split[0]
splits = found[SPLITS_CSV]
classes = json.loads((SPLITS_CSV.parent / 'classes.json').read_text())
K = len(classes)
COLOR_DIR = find_color_dir(INPUT_ROOT)
train = splits[splits['split'] == 'train'].reset_index(drop=True)
train_counts = train['label'].value_counts()
ORDER = [int(i) for i in train_counts.index]   # from the most to the fewest training images
need = {i: max(0, FILL_TARGET - int(train_counts.get(i, 0))) for i in range(K)}
print('Split: ', SPLITS_CSV)
print('Images:', COLOR_DIR)
print(pd.DataFrame({'train images': [int(train_counts[i]) for i in ORDER], 'to generate': [need[i] for i in ORDER]},
                   index=[classes[i]['name'] for i in ORDER]).to_string())
print(f'Total to generate: {sum(need.values()):,}')

# The label-check model: notebook 02's final ResNet50 (the best Phase 2 model)
checkpoints = []
for f in find_files(INPUT_ROOT, 'resnet50.pt'):
    ckpt = torch.load(f, map_location='cpu', weights_only=True)
    if ckpt.get('run') == 'final':
        checkpoints.append((f, ckpt))
if not checkpoints:
    raise FileNotFoundError('models/resnet50.pt of notebook 02\'s final run was not found. Add Input → Your Work → '
                            'Notebooks → notebook 02, the version with FINAL = True.')
CLS_PATH, cls_ckpt = checkpoints[0]
assert [c['folder'] for c in cls_ckpt['classes']] == [c['folder'] for c in classes], 'class lists do not match'
print(f'\nLabel check: {cls_ckpt["name"]}, seed {cls_ckpt["seed"]}, validation macro-F1 '
      f'{100 * cls_ckpt["metrics"]["validation_macro_F1"]:.2f}%  ({CLS_PATH})')

# The GAN: part 1's newest progress.json, and the snapshot it names
progress = []
for f in find_files(INPUT_ROOT, 'progress.json'):
    try:
        p = json.loads(f.read_text())
    except (OSError, ValueError):
        continue
    if p.get('kind') == PROGRESS_KIND:
        progress.append(p | {'folder': str(f.parent)})
if not progress:
    raise FileNotFoundError('gan/progress.json of notebook 03_gan_train was not found. Add Input → Your Work → '
                            'Notebooks → 03_gan_train, its latest version.')
GAN = max(progress, key=lambda p: (p['kimg'], p['session']))
SNAPSHOT = Path(GAN['folder']) / GAN[SNAPSHOT_CHOICE]['file']
end_kimg = int(GAN[SNAPSHOT_CHOICE]['kimg'])
if not SNAPSHOT.exists():
    raise FileNotFoundError(f'{SNAPSHOT} is missing from part 1\'s output.')
assert GAN['split_fingerprint'] == SPLIT_FINGERPRINT and GAN['settings']['res'] == RES
print(f'\nGAN: {SNAPSHOT_CHOICE} snapshot {SNAPSHOT.name} ({end_kimg:,} kimg, FID {GAN[SNAPSHOT_CHOICE]["fid"]:.2f}), '
      f'trained {GAN["session"]} session(s), {GAN["train_hours_total"]} GPU hours in all')

## 2. The GAN code

A snapshot holds the networks together with their program code, which needs NVIDIA's helper code to run. So the same
version of the `stylegan3` repository is downloaded, with the same changes as in part 1 (see its section 2).

In [ ]:
REPO = WORK_DIR / 'stylegan3'

def run(*command):
    subprocess.run([str(c) for c in command], check=True)

def pip_install(*packages):
    run(sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *packages)

for module, package in [('pytorch_fid', 'pytorch-fid==0.3.0'), ('psutil', 'psutil'), ('click', 'click')]:
    if importlib.util.find_spec(module) is None:
        pip_install(package)
if DEVICE.type == 'cuda' and shutil.which('ninja') is None:   # ninja: needed to compile the CUDA programs
    pip_install('ninja')
    os.environ['PATH'] = str(Path(sys.executable).parent) + os.pathsep + os.environ['PATH']

# The changes of part 1 (03_gan_train, section 2), the same list
PATCHES = [
    ('dnnlib/util.py',
     'from distutils.util import strtobool',
     "def strtobool(text): return text.strip().lower() in ('y', 'yes', 't', 'true', 'on', '1')  # changed: no distutils"),
    ('torch_utils/ops/conv2d_gradfix.py', 'from pkg_resources import parse_version\n', ''),
    ('torch_utils/ops/conv2d_gradfix.py',
     "_use_pytorch_1_11_api = parse_version(torch.__version__) >= parse_version('1.11.0a') # Allow prerelease builds of 1.11",
     '_use_pytorch_1_11_api = True  # changed: PyTorch 2.x'),
    ('torch_utils/ops/grid_sample_gradfix.py', 'from pkg_resources import parse_version\n', ''),
    ('torch_utils/ops/grid_sample_gradfix.py',
     "_use_pytorch_1_11_api = parse_version(torch.__version__) >= parse_version('1.11.0a') # Allow prerelease builds of 1.11",
     '_use_pytorch_1_11_api = True  # changed: PyTorch 2.x'),
    ('torch_utils/ops/grid_sample_gradfix.py',
     "_use_pytorch_1_12_api = parse_version(torch.__version__) >= parse_version('1.12.0a') # Allow prerelease builds of 1.12",
     '_use_pytorch_1_12_api = True'),
    ('torch_utils/ops/grid_sample_gradfix.py',
     "        op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')\n"
     "        if _use_pytorch_1_12_api:\n"
     "            op = op[0]\n",
     '        op = torch.ops.aten.grid_sampler_2d_backward  # changed: the public way to call it\n'),
    ('torch_utils/misc.py', '        super().__init__(dataset)\n',
     '        super().__init__()  # changed: newer PyTorch takes no argument here\n'),
    ('training/training_loop.py',
     '    gw = np.clip(7680 // training_set.image_shape[2], 7, 32)',
     '    gw = 8  # changed: a smaller sample grid'),
    ('training/training_loop.py',
     '    gh = np.clip(4320 // training_set.image_shape[1], 4, 32)',
     '    gh = max(training_set.label_dim, 4)  # one row per class'),
]

marker = REPO / 'PATCHED_BY_NOTEBOOK_03'
if not marker.exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    run('git', 'clone', '--quiet', STYLEGAN_REPO, REPO)
    run('git', '-C', REPO, 'checkout', '--quiet', STYLEGAN_COMMIT)
    for rel_path, old, new in PATCHES:
        file = REPO / rel_path
        text = file.read_text(encoding='utf-8')
        if text.count(old) != 1:
            raise RuntimeError(f'{rel_path}: the code to change was found {text.count(old)} times, not once:\n{old}')
        file.write_text(text.replace(old, new, 1), encoding='utf-8')
    marker.write_text(STYLEGAN_COMMIT)
sys.path.insert(0, str(REPO))

# The two CUDA programs: compiled if possible, otherwise plain PyTorch (the same images, a little slower)
CUDA_KERNELS = False
if DEVICE.type == 'cuda':
    from torch_utils.ops import bias_act, upfirdn2d
    try:
        x = torch.randn(2, 4, 8, 8, device=DEVICE)
        bias_act.bias_act(x, torch.zeros(4, device=DEVICE), act='lrelu', impl='cuda')
        upfirdn2d.upsample2d(x, upfirdn2d.setup_filter([1, 3, 3, 1], device=DEVICE), impl='cuda')
        torch.cuda.synchronize()
        CUDA_KERNELS = bias_act._plugin is not None and upfirdn2d._plugin is not None
    except Exception as e:
        print(f'The CUDA programs did not compile ({type(e).__name__}: {str(e)[-300:]})')
    if not CUDA_KERNELS:
        bias_act._init = upfirdn2d._init = lambda: False
print(f'StyleGAN code: commit {STYLEGAN_COMMIT[:7]}, {len(PATCHES)} changes; CUDA programs '
      f'{"compiled" if CUDA_KERNELS else "not used"}')

## 3. Generate images for the small classes

The generator **G_ema** (the smoothed generator StyleGAN2-ADA keeps for drawing images) draws images of one class at
a time. Each image is saved as a JPEG (quality 95, like the PlantVillage photos), read back and resized to 224 × 224
exactly as notebooks 02 and 04 read a photo, and then shown to the label-check model. **An image is kept only if the
model names its intended class** (check 4 of the plan). Images are generated until the class reaches 2,000, or until
3 × the needed number were tried. Each image comes from a fixed seed, so the set can be made again.

In [ ]:
with open(SNAPSHOT, 'rb') as f:
    G = pickle.load(f)['G_ema'].to(DEVICE).eval().requires_grad_(False)
assert G.c_dim == K and G.img_resolution == RES, (G.c_dim, G.img_resolution)

@torch.no_grad()
def generate(label, seeds):
    """Images of one class, (n, RES, RES, 3) uint8; seed s always gives the same image."""
    z = torch.from_numpy(np.stack([np.random.RandomState(s).randn(G.z_dim) for s in seeds])).float().to(DEVICE)
    c = torch.zeros(len(seeds), G.c_dim, device=DEVICE)
    c[:, label] = 1
    img = G(z, c, truncation_psi=TRUNCATION_PSI, noise_mode='const').float()
    return (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()

print(f'Generator loaded from {SNAPSHOT.name}')

In [ ]:
def build_classifier(ckpt):
    """A notebook 02 checkpoint as a network (the last layer replaced as in notebook 02 and the web app)."""
    arch = ckpt['arch']
    if arch == 'resnet50':
        net = models.resnet50(weights=None)
        net.fc = nn.Linear(net.fc.in_features, K)
    elif arch == 'mobilenet_v2':
        net = models.mobilenet_v2(weights=None)
        net.classifier[1] = nn.Linear(net.last_channel, K)
    elif arch == 'vgg16':
        net = models.vgg16(weights=None)
        net.classifier[6] = nn.Linear(net.classifier[6].in_features, K)
    else:
        raise ValueError(f'unsupported label-check model {arch}')
    net.load_state_dict(ckpt['state_dict'])
    return net.to(DEVICE).eval()

classifier = build_classifier(cls_ckpt)
CLS_SIZE = int(cls_ckpt.get('img_size', 224))
CLS_MEAN = torch.tensor(cls_ckpt['mean'], device=DEVICE).view(1, 3, 1, 1) * 255
CLS_STD = torch.tensor(cls_ckpt['std'], device=DEVICE).view(1, 3, 1, 1) * 255

def to_jpeg(pixels):
    buf = BytesIO()
    Image.fromarray(pixels).save(buf, format='JPEG', quality=JPEG_QUALITY)
    return buf.getvalue()

def classifier_pixels(jpeg):
    """Read an image as notebooks 02 and 04 do: centre square, bilinear resize to CLS_SIZE."""
    with Image.open(BytesIO(jpeg)) as img:
        img = img.convert('RGB')
        side = min(img.size)
        left, top = (img.width - side) // 2, (img.height - side) // 2
        img = img.crop((left, top, left + side, top + side))
        return np.asarray(img.resize((CLS_SIZE, CLS_SIZE), Image.Resampling.BILINEAR))

@torch.inference_mode()
def classify(pixels):
    x = (torch.from_numpy(np.stack(pixels)).to(DEVICE).permute(0, 3, 1, 2).float() - CLS_MEAN) / CLS_STD
    return torch.softmax(classifier(x).float(), dim=1).cpu().numpy()

if SYNTH_OUT.exists():
    shutil.rmtree(SYNTH_OUT)
records, rejected_examples = [], {}
start = time.time()
for label in ORDER:
    if need[label] == 0:
        continue
    folder = classes[label]['folder']
    (SYNTH_OUT / folder).mkdir(parents=True, exist_ok=True)
    kept, tried, next_seed = 0, 0, label * 1_000_000
    while kept < need[label] and tried < MAX_TRIES * need[label]:
        seeds = list(range(next_seed, next_seed + GEN_BATCH))
        next_seed += GEN_BATCH
        jpegs = [to_jpeg(p) for p in generate(label, seeds)]
        probs = classify([classifier_pixels(j) for j in jpegs])
        for s, j, pr in zip(seeds, jpegs, probs):
            predicted = int(pr.argmax())
            passed = predicted == label
            keep = passed and kept < need[label]
            if keep:
                (SYNTH_OUT / folder / f'{s:07d}.jpg').write_bytes(j)
                kept += 1
            elif not passed and len(rejected_examples.setdefault(label, [])) < 8:
                rejected_examples[label].append((j, predicted, float(pr[predicted])))
            records.append({'label': label, 'seed': s, 'predicted': predicted, 'prob_intended': float(pr[label]),
                            'passed': passed, 'kept': keep})
        tried += len(seeds)
    print(f'{classes[label]["name"]:24s} kept {kept:5,} of {need[label]:5,} needed  ({tried:,} generated)')
print(f'Generated and checked in {(time.time() - start) / 60:.1f} minutes')

gen = pd.DataFrame(records)
synthetic = gen[gen['kept']].copy()
synthetic['class_folder'] = synthetic['label'].map(lambda i: classes[i]['folder'])
synthetic['class_name'] = synthetic['label'].map(lambda i: classes[i]['name'])
synthetic['path'] = synthetic['class_folder'] + '/' + synthetic['seed'].map(lambda s: f'{s:07d}.jpg')
synthetic = synthetic[['path', 'class_folder', 'class_name', 'label', 'seed', 'prob_intended']]
synthetic.to_csv(SYNTH_OUT / 'synthetic.csv', index=False)
gen.to_csv(RESULTS_OUT / '03_generated_all.csv', index=False)
print(f'\n{len(synthetic):,} images kept in {SYNTH_OUT}')

## 4. The checks

1. **Sample sheet, by eye:** real and generated leaves of each class side by side. Look for blurry or broken leaves,
   a wrong background, or symptoms that do not belong to the class.
2. **FID and KID per class:** how close the generated images are to the real training images of the same class, in
   the features of an Inception network (lower is better). KID is the one to compare between classes: it does not
   depend on the number of images, while FID is too high for small classes (Mosaic virus has only 261 real images).
   The *real vs real* value (one half of the class's training images against the other half) shows the best a GAN
   could reach.
3. **Nearest real image:** a GAN that copies its training images would make useless "new" images. For every generated
   image, the most similar training image (cosine similarity of the Inception features) is found. For comparison,
   the same is done for the validation images, which are new leaves: if generated images were much closer to the
   training images than new real leaves are, the GAN would be copying.
4. **Label check:** the share of generated images that ResNet50 recognised as their class, and what the others were
   taken for.

In [ ]:
def pick(df, n, seed):
    return df.sample(n=min(n, len(df)), random_state=seed) if len(df) else df

def read_rgb(path, size=None):
    with Image.open(path) as img:
        img = img.convert('RGB')
        return np.asarray(img.resize((size, size), Image.Resampling.BILINEAR) if size else img)

def sheet(rows, row_labels, title, path, thumb=112, gap=4):
    """A grid of images, one labelled row each."""
    cols = max(len(r) for r in rows)
    canvas = np.full((len(rows) * (thumb + gap), cols * (thumb + gap), 3), 252, np.uint8)
    for i, row in enumerate(rows):
        for j, img in enumerate(row):
            small = np.asarray(Image.fromarray(img).resize((thumb, thumb), Image.Resampling.LANCZOS))
            canvas[i * (thumb + gap):i * (thumb + gap) + thumb, j * (thumb + gap):j * (thumb + gap) + thumb] = small
    fig, ax = plt.subplots(figsize=(cols * 0.95 + 2.2, len(rows) * 0.95 + 0.8))
    ax.imshow(canvas)
    ax.set_yticks([i * (thumb + gap) + thumb / 2 for i in range(len(rows))], row_labels)
    ax.set_xticks([])
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title)
    fig.savefig(path)
    plt.show()

GEN_CLASSES = [i for i in ORDER if need[i] > 0 and (synthetic['label'] == i).any()]
rows, labels = [], []
for i in GEN_CLASSES:
    real_rows = pick(train[train['label'] == i], 8, SEED)
    gen_rows = pick(synthetic[synthetic['label'] == i], 8, SEED)
    rows += [[read_rgb(COLOR_DIR / p) for p in real_rows['path']], [read_rgb(SYNTH_OUT / p) for p in gen_rows['path']]]
    labels += [f'{classes[i]["name"]}: real', 'generated']
sheet(rows, labels, f'Check 1: real (upper row) and generated (lower row) leaves, {end_kimg:,} kimg',
      RESULTS_OUT / '03_real_vs_generated.png')

In [ ]:
from pytorch_fid.inception import InceptionV3

inception = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(DEVICE).eval()

@torch.inference_mode()
def features(paths, batch=100):
    """2048 Inception features per image (the standard FID network; it resizes the images to 299 px itself)."""
    out = []
    with ThreadPoolExecutor(8) as pool:
        for i in range(0, len(paths), batch):
            imgs = np.stack(list(pool.map(read_rgb, paths[i:i + batch])))
            x = torch.from_numpy(imgs).to(DEVICE).permute(0, 3, 1, 2).float() / 255
            out.append(inception(x)[0].flatten(1).double().cpu())
    return torch.cat(out).numpy()

start = time.time()
val = splits[splits['split'] == 'val'].reset_index(drop=True)
train_f = features([COLOR_DIR / p for p in train['path']])
val_f = features([COLOR_DIR / p for p in val['path']])
synth_f = features([SYNTH_OUT / p for p in synthetic['path']])
print(f'Inception features of {len(train_f) + len(val_f) + len(synth_f):,} images in {time.time() - start:.0f} s')

def fid(a, b):
    """Fréchet distance between two feature sets (FID)."""
    mu_a, mu_b = a.mean(0), b.mean(0)
    cov_a, cov_b = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    w, v = np.linalg.eigh(cov_a)
    sqrt_a = (v * np.sqrt(w.clip(0))) @ v.T
    tr_covmean = np.sqrt(np.linalg.eigvalsh(sqrt_a @ cov_b @ sqrt_a).clip(0)).sum()   # trace of sqrtm(cov_a @ cov_b)
    return float(((mu_a - mu_b) ** 2).sum() + np.trace(cov_a) + np.trace(cov_b) - 2 * tr_covmean)

def kid(a, b, subsets=100, subset_size=1000, seed=0):
    """Kernel Inception distance (unbiased, cubic polynomial kernel, mean of random subsets), as in StyleGAN2-ADA."""
    rng = np.random.default_rng(seed)
    n = min(len(a), len(b), subset_size)
    A, B = torch.tensor(a, device=DEVICE, dtype=torch.float64), torch.tensor(b, device=DEVICE, dtype=torch.float64)
    dev = DEVICE
    d = a.shape[1]
    values = []
    for _ in range(subsets):
        x = A[torch.from_numpy(rng.choice(len(a), n, replace=False)).to(dev)]
        y = B[torch.from_numpy(rng.choice(len(b), n, replace=False)).to(dev)]
        kxx, kyy, kxy = (x @ x.T / d + 1) ** 3, (y @ y.T / d + 1) ** 3, (x @ y.T / d + 1) ** 3
        t = (kxx.sum() - kxx.diagonal().sum() + kyy.sum() - kyy.diagonal().sum()) / (n - 1) - kxy.sum() * 2 / n
        values.append(float(t) / n)
    return float(np.mean(values)), float(np.std(values))

check_rows = []
rng = np.random.default_rng(SEED)
for i in GEN_CLASSES:
    real_i = train_f[(train['label'] == i).to_numpy()]
    gen_i = synth_f[(synthetic['label'] == i).to_numpy()]
    half = rng.permutation(len(real_i))
    a, b = real_i[half[:len(half) // 2]], real_i[half[len(half) // 2:]]
    kid_gen, kid_gen_sd = kid(real_i, gen_i)
    kid_real, _ = kid(a, b)
    check_rows.append({'class': classes[i]['name'], 'label': i, 'train images': len(real_i), 'kept': len(gen_i),
                       'FID': fid(real_i, gen_i), 'FID real vs real': fid(a, b),
                       'KID x1000': 1000 * kid_gen, 'KID x1000 sd': 1000 * kid_gen_sd,
                       'KID x1000 real vs real': 1000 * kid_real})
checks = pd.DataFrame(check_rows)
print('Check 2: FID and KID per class (lower is better; real vs real = one half of the training images against the other)\n')
print(checks[['class', 'train images', 'kept', 'FID', 'FID real vs real', 'KID x1000', 'KID x1000 real vs real']]
      .round(2).to_string(index=False))
print(f'\nMean over the classes: FID {checks["FID"].mean():.1f}, KID x1000 {checks["KID x1000"].mean():.2f}')

fig, ax = plt.subplots(figsize=(8, 0.45 * len(checks) + 1.4))
y = np.arange(len(checks))
ax.barh(y, checks['KID x1000'], height=0.6, color=BLUE, label='generated vs real')
ax.errorbar(checks['KID x1000'], y, xerr=checks['KID x1000 sd'], fmt='none', ecolor=INK_2, elinewidth=1, capsize=2)
ax.plot(checks['KID x1000 real vs real'].clip(lower=0), y, '|', color=ORANGE, markersize=16, markeredgewidth=2.5,
        label='real vs real (best reachable)')
ax.set_yticks(y, checks['class'])
ax.invert_yaxis()
ax.tick_params(axis='y', length=0)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.set_xlabel('KID × 1000 (lower is better)')
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.set_title(f'Check 2: KID per class, {end_kimg:,} kimg')
fig.savefig(RESULTS_OUT / '03_kid_per_class.png')
plt.show()

In [ ]:
def unit(f):
    return f / np.linalg.norm(f, axis=1, keepdims=True)

nn_rows, pairs = [], []
for i in GEN_CLASSES:
    t_mask = (train['label'] == i).to_numpy()
    t_f = unit(train_f[t_mask])
    t_paths = train.loc[t_mask, 'path'].to_numpy()
    g_sim_all = unit(synth_f[(synthetic['label'] == i).to_numpy()]) @ t_f.T
    v_sim = (unit(val_f[(val['label'] == i).to_numpy()]) @ t_f.T).max(1)
    g_sim = g_sim_all.max(1)
    nn_rows.append({'class': classes[i]['name'], 'generated: median similarity': np.median(g_sim),
                    'new real leaves (val): median similarity': np.median(v_sim),
                    'generated closer than 99% of new leaves %': 100 * np.mean(g_sim > np.percentile(v_sim, 99)),
                    'g_sim': g_sim, 'v_sim': v_sim})
    gen_paths = synthetic.loc[synthetic['label'] == i, 'path'].to_numpy()
    for k in np.random.default_rng(i).choice(len(gen_paths), size=min(4, len(gen_paths)), replace=False):
        pairs.append((i, read_rgb(SYNTH_OUT / gen_paths[k]), read_rgb(COLOR_DIR / t_paths[g_sim_all[k].argmax()]),
                      g_sim[k]))
nn_table = pd.DataFrame(nn_rows)
print('Check 3: similarity to the most similar training image (1 = identical features)\n')
print(nn_table.drop(columns=['g_sim', 'v_sim']).round(3).to_string(index=False))
print('\nIf the GAN copied, many generated images would be closer to a training image than 99% of the new real leaves '
      '(far more than 1%), and the pairs below would show the same leaf twice.')

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(min(np.concatenate(nn_table['g_sim']).min(), np.concatenate(nn_table['v_sim']).min()), 1, 40)
ax.hist(np.concatenate(nn_table['v_sim']), bins=bins, density=True, color=ORANGE, alpha=0.55, label='new real leaves (val)')
ax.hist(np.concatenate(nn_table['g_sim']), bins=bins, density=True, histtype='step', color=BLUE, linewidth=2,
        label='generated')
ax.set_xlabel('Similarity to the most similar training image of the same class')
ax.set_ylabel('Share of images')
ax.set_yticks([])
ax.legend(frameon=False)
ax.set_title('Check 3: are the generated images copies?')
fig.savefig(RESULTS_OUT / '03_nearest_similarity.png')
plt.show()

rows, labels = [], []
for i in GEN_CLASSES:
    ps = [p for p in pairs if p[0] == i]
    rows.append([img for _, g, t, _ in ps for img in (g, t)])
    labels.append(classes[i]['name'])
sheet(rows, labels, 'Check 3: generated image, then its most similar training image (4 pairs per class)',
      RESULTS_OUT / '03_nearest_real.png')

In [ ]:
label_check = gen.groupby('label').agg(generated=('seed', 'size'), passed=('passed', 'sum'), kept=('kept', 'sum'))
label_check['passed %'] = 100 * label_check['passed'] / label_check['generated']
label_check['needed'] = [need[i] for i in label_check.index]
label_check['short'] = label_check['needed'] - label_check['kept']
mistaken = gen[~gen['passed']].groupby('label')['predicted'].agg(
    lambda s: ', '.join(f'{classes[c]["name"]} {n}' for c, n in s.value_counts().head(2).items()))
label_check['most often taken for'] = mistaken.reindex(label_check.index).fillna('')
label_check.index = [classes[i]['name'] for i in label_check.index]
label_check = label_check.loc[[classes[i]['name'] for i in ORDER if classes[i]['name'] in label_check.index]]
print('Check 4: label check by', cls_ckpt['name'], '\n')
print(label_check.round(1).to_string())
if label_check['short'].sum() > 0:
    print(f'\n{int(label_check["short"].sum()):,} images short of the fill target: these classes stay below '
          f'{FILL_TARGET:,}. Notebook 04 uses what there is.')

rows, labels = [], []
for i in ORDER:
    if rejected_examples.get(i):
        rows.append([np.asarray(Image.open(BytesIO(j)).convert('RGB')) for j, _, _ in rejected_examples[i][:6]])
        labels.append(f'{classes[i]["name"]} → ' + ', '.join(classes[p]['name'] for _, p, _ in rejected_examples[i][:2]))
if rows:
    sheet(rows, labels, 'Check 4: dropped images (intended class → what ResNet50 saw in the first two)',
          RESULTS_OUT / '03_dropped_examples.png')

## 5. Save

In [ ]:
per_class = checks.drop(columns=['label']).merge(nn_table.drop(columns=['g_sim', 'v_sim']), on='class')
per_class = per_class.merge(label_check[['generated', 'passed %', 'short']], left_on='class', right_index=True)
per_class.to_csv(RESULTS_OUT / '03_per_class.csv', index=False)

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'gan': 'StyleGAN2-ADA (NVlabs/stylegan3, stylegan2 configuration), class-conditional, trained on the train split only',
    'code_commit': STYLEGAN_COMMIT, 'resolution': RES, 'kimg': end_kimg, 'snapshot': SNAPSHOT.name,
    'snapshot_choice': SNAPSHOT_CHOICE, 'snapshot_fid_overall': GAN[SNAPSHOT_CHOICE]['fid'],
    'gan_sessions': GAN['session'], 'gan_train_hours': GAN['train_hours_total'], 'gan_settings': GAN['settings'],
    'fill_target': FILL_TARGET, 'truncation_psi': TRUNCATION_PSI, 'jpeg_quality': JPEG_QUALITY,
    'kept_images': int(len(synthetic)), 'fid_mean_per_class': float(checks['FID'].mean()),
    'kid_x1000_mean_per_class': float(checks['KID x1000'].mean()),
    'label_check_model': {'name': cls_ckpt['name'], 'seed': cls_ckpt['seed'],
                          'rule': 'kept if its top class is the intended class'},
    'split_fingerprint': SPLIT_FINGERPRINT,
    'per_class': per_class.to_dict(orient='records'),
}
(RESULTS_OUT / '03_summary.json').write_text(json.dumps(summary, indent=1, default=float))

print(f'{SNAPSHOT.name} ({end_kimg:,} kimg): {len(synthetic):,} images kept; mean per class: FID '
      f'{summary["fid_mean_per_class"]:.1f}, KID x1000 {summary["kid_x1000_mean_per_class"]:.2f}; '
      f'{(time.time() - NOTEBOOK_START) / 60:.0f} minutes\n')
for f in sorted(RESULTS_OUT.iterdir()):
    print(f'{f.relative_to(OUT_DIR)}  ({f.stat().st_size / 1e6:,.1f} MB)')
print(f'synthetic/  ({len(synthetic):,} images, {sum(f.stat().st_size for f in SYNTH_OUT.rglob("*.jpg")) / 1e6:,.0f} MB)')

## Checklist

1. **Sample sheet** (`03_real_vs_generated.png`): the generated leaves should look like real ones of the same class:
   one leaf on a grey background, the typical spots or discolouration of the disease. Note classes that look wrong.
2. **KID** (`03_kid_per_class.png`): classes far above their *real vs real* value are the weak ones. Per-class FID is
   in the table too; it is high for the small classes just because they have few real images.
3. **Copies** (`03_nearest_real.png`, `03_nearest_similarity.png`): the pairs should show different leaves, and the
   blue outline should not sit far to the right of the orange area.
4. **Label check**: a pass rate far below the others means the GAN draws that class poorly. Classes that stay short of
   2,000 are fine for notebook 04, but note them.

**For the report:** the sample sheet, the KID chart with the table, the label check table, the nearest-image sheet,
and from part 1 the FID curve and the GPU hours.

**Next:** notebook 04 reads `synthetic/synthetic.csv` from this notebook's output (Add Input → Your Work →
Notebooks → 03_gan_generate).